In [1]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import v2

DATA_ROOT = Path("data")
SPLIT_PATH = Path("split_indices_seed42.pth")
SPLIT_SEED = 42
BATCH_SIZE = 64

In [2]:
base_dataset = datasets.EuroSAT(
    root=DATA_ROOT, download=False, transform=None
)

class_names = base_dataset.classes
print(f"Images: {len(base_dataset)}")
print(f"Classes: {class_names}")

Images: 27000
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


In [4]:
# We want augmentation only for training, not for validation or test. With the seed, it gives indices for each of the datasets. So we take only those particular images from the training dataset and evaluation dataset so that there is no overlap or anything.

train_size = int(0.7 * len(base_dataset))
validation_size = int(0.2 * len(base_dataset))
test_size = len(base_dataset) - train_size - validation_size

if SPLIT_PATH.exists():
    split_indices = torch.load(SPLIT_PATH, weights_only=True)
    print("Loaded the existing split.")
else:
    generator = torch.Generator().manual_seed(SPLIT_SEED)
    train_split, validation_split, test_split = torch.utils.data.random_split(
        base_dataset,
        [train_size, validation_size, test_size],
        generator=generator,
    )
    split_indices = {
        "train": train_split.indices,
        "validation": validation_split.indices,
        "test": test_split.indices,
    }
    torch.save(split_indices, SPLIT_PATH)
    print("Created and saved the split.")

print({name: len(indices) for name, indices in split_indices.items()})

Loaded the existing split.
{'train': 18900, 'validation': 5400, 'test': 2700}


In [5]:
#augmentation is only for training set
MODEL_TYPE = globals().get("MODEL_TYPE", "resnet18")  # May be set by a calling notebook

if MODEL_TYPE == "custom_cnn":
    train_transform = v2.Compose([
        v2.ToImage(),
        v2.RandomHorizontalFlip(),
        v2.RandomVerticalFlip(),
        v2.RandomRotation(20),
        v2.ToDtype(torch.float32, scale=True),
    ])
    eval_transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ])

elif MODEL_TYPE == "resnet18":

    #image normalisation 
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]

    train_transform = v2.Compose([
        v2.ToImage(),
        v2.RandomHorizontalFlip(),
        v2.RandomVerticalFlip(),
        v2.RandomRotation(20),
        v2.ToDtype(torch.float32, scale=True), #replacement of toTensor
        v2.Normalize(mean=imagenet_mean, std=imagenet_std),
    ])
    eval_transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=imagenet_mean, std=imagenet_std),
    ])

else:
    raise ValueError("MODEL_TYPE must be 'custom_cnn' or 'resnet18'.")

## 4. Create datasets and dataloaders

In [6]:
train_dataset_full = datasets.EuroSAT(
    root=DATA_ROOT, download=False, transform=train_transform
)
eval_dataset_full = datasets.EuroSAT(
    root=DATA_ROOT, download=False, transform=eval_transform
)

#Take subset of dataset with particular indices as decided by the seed
training_data = Subset(train_dataset_full, split_indices["train"])
validation_data = Subset(eval_dataset_full, split_indices["validation"])
testing_data = Subset(eval_dataset_full, split_indices["test"])

train_dataloader = DataLoader(
    training_data, batch_size=BATCH_SIZE, shuffle=True
)
validate_dataloader = DataLoader(
    validation_data, batch_size=BATCH_SIZE, shuffle=False
)
test_dataloader = DataLoader(
    testing_data, batch_size=BATCH_SIZE, shuffle=False
)

print(f"Using the {MODEL_TYPE} pipeline.")
print(f"Train: {len(training_data)}")
print(f"Validation: {len(validation_data)}")
print(f"Test: {len(testing_data)}")

Using the resnet18 pipeline.
Train: 18900
Validation: 5400
Test: 2700


## 5. Check one batch

In [7]:
images, labels = next(iter(train_dataloader))

print(f"Image batch: {images.shape}") #batch size, channels, w, h
print(f"Label batch: {labels.shape}")
print(f"Data type: {images.dtype}")

Image batch: torch.Size([64, 3, 64, 64])
Label batch: torch.Size([64])
Data type: torch.float32
